In [1]:
import pickle
import sys
import copy
import time
import os

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

from utils.parameters import human_model as m_model



mu_val = 1e-9
n_cores = 10
base = 0
counter = 5
lp_path = '/data2/hratch/human_me/other/test_lp/'

No objective coefficients in model. Unclear what should be optimized


In [2]:
def add_sink(m, tme_new = None):
    '''m is a cobra.metabolite or metabolite ID'''
    
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    
    if isinstance(m, cobra.Metabolite): # object
        m_id = m.id
    else: # string
        m_id = m
    
    tme_new.add_boundary(tme_new.metabolites.get_by_id(m_id), type ='sink')
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    
    return tme_new, sln, stat

In [9]:
with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme0 = pickle.load(handle)

In [10]:
sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)

Getting MINOS parameters...
Done in 108.456 seconds with status 1


In [11]:
tme1, sln1, stat1 = add_sink(m = 'adp_c')

Getting MINOS parameters...
Done in 104.851 seconds with status 1


In [10]:
flux_res = pd.DataFrame(index = [r.id for r in tme0.reactions], columns = ['flux0', 'flux1'])
for i in flux_res.index:
    flux_res.loc[i,:] = [sln0[tme0.reactions.index(i)], sln1[tme1.reactions.index(i)]]
flux_res['1-0'] = flux_res.flux1 - flux_res.flux0
flux_res['abs_difference'] = flux_res['1-0'].abs()
flux_res.sort_values(by = 'abs_difference', ascending = False, inplace = True)

flux_res = flux_res[flux_res.abs_difference > 1e-25]

In [ ]:
flux_res.head(20)

In [23]:
flux_res.head(20) 

,flux0,flux1,1-0,abs_difference
SRTNtu_R_0,0.00011413,0.000216672,0.000102542,0.000102542
SRTNtu_F_0,0.00011413,0.000215916,0.000101786,0.000101786
CLCFTRte_F,6.42315e-43,3.05433e-06,3.05433e-06,3.05433e-06
NKCCt_F_0,0,1.51135e-06,1.51135e-06,1.51135e-06
r1116_R,2.88211e-10,1.50224e-06,1.50195e-06,1.50195e-06
EX_atp_LPAREN_e_RPAREN_,-2.88211e-10,-1.50224e-06,-1.50195e-06,1.50195e-06
EX_atp_b,-2.88211e-10,-1.50224e-06,-1.50195e-06,1.50195e-06
PIt7_R_0,4.14884e-08,1.54184e-06,1.50035e-06,1.50035e-06
EX_pi_LPAREN_e_RPAREN_,4.14884e-08,1.54184e-06,1.50035e-06,1.50035e-06
EX_pi_b,4.14884e-08,1.54184e-06,1.50035e-06,1.50035e-06


In [6]:
tme0.reactions.get_by_id('SRTNtu_R_0').reaction

'1.5321763430960728e-06 HGNC:10963_enzyme_deg_proxy + 8.35313490331122e5*mu  1.53217634309607e6 HGNC:10963_folded_protein_c + srtn_c --> srtn_e'

In [25]:
tme0.reactions.get_by_id('SRTNtu_R_0').reaction

'8.58367144551372e5*mu  1.57446257937388e6 HGNC:10963_folded_protein_e + srtn_c --> srtn_e'

In [27]:
tme0.reactions.get_by_id('SRTNtu_F_0').reaction

'8.58367144551372e5*mu  1.57446257937388e6 HGNC:10963_folded_protein_e + srtn_e --> srtn_c'

In [28]:
tme0.reactions.get_by_id('CLCFTRte_F').reaction

'3.94095804360994e5*mu  7.2287144329008e7 HGNC:1884_folded_protein_e + cl_c --> cl_e'

In [29]:
tme0.reactions.get_by_id('NKCCt_F_0').reaction

'5.04902999317694e5*mu  9.26119882017187e7 HGNC:10910_folded_protein_e + 2.0 cl_e + k_e + na1_e --> 2.0 cl_c + k_c + na1_c'

In [30]:
tme0.reactions.get_by_id('r1116_R').reaction

'6.25345638024967e5*mu  1.05500572399119e6 HGNC:DUMMY_folded_protein_c + atp_e --> atp_c'

In [32]:
tme0.reactions.get_by_id('PIt7_R_0').reaction

'9.8871723548704e5*mu  1.81355763525869e6 HGNC:10929_folded_protein_e + 3.0 na1_c + pi_c --> 3.0 na1_e + pi_e'

In [33]:
tme0.reactions.get_by_id('H2Ot_F_0')

Reaction identifier,H2Ot_F_0
Name,H2O transport via diffusion
Memory address,0x07efbe86f9390
Stoichiometry,0.000162688423693954*mu 2.98411742375495e6 HGNC:642_folded_protein_e + h2o_e --> h2o_c 0.000162688423693954*mu 2.98411742375495e6 + water --> water
GPR,HGNC:633 or HGNC:634 or HGNC:637 or HGNC:638 or HGNC:642 or HGNC:7103
Lower bound,0
Upper bound,inf
